## Section 1: Importing and Loading Data
- numpy → handles numerical operations efficiently (arrays, matrices, etc.)
- pandas → for data manipulation and analysis using DataFrames.

- pd.read_csv("bitcoin.csv") → loads your Bitcoin dataset into a DataFrame.
- df.head() → shows the first 5 rows, just to get a peek at the structure.

In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv("bitcoin.csv")
df.head()

,Date,Price
0,5/23/2019,7881.846680
1,5/24/2019,7987.371582
2,5/25/2019,8052.543945
3,5/26/2019,8673.215820
4,5/27/2019,8805.778320


## Section 2: Dropping Unnecessary Columns
- Removes the Date column since SVM can’t directly understand date strings.
- axis=1 means “drop column” (axis=0 would mean “drop row”).
- inplace=True applies the change directly to df.

In [2]:
df.drop(['Date'],axis=1,inplace=True)

## Section 3: Creating the Prediction Target
- You’re telling the model: “Predict what the Price will be 30 days into the future.
- .shift(-30) moves the column upward by 30 rows.
- Example: The value in row 0’s Prediction = value of row 30’s Price.

In [3]:
predictionDays = 30
df['Prediction'] = df[['Price']].shift(-predictionDays)
df.head()

,Price,Prediction
0,7881.846680,10701.69141
1,7987.371582,10855.37109
2,8052.543945,11011.10254
3,8673.215820,11790.91699
4,8805.778320,13016.23145


## Section 4: Inspecting the Tail
- Displays the last 5 rows.
- You’ll notice the last 30 rows of Prediction are NaN — because there’s no data 30 days after the last record to fill them.

In [4]:
df.tail()

,Price,Prediction
362,9729.038086,NaN
363,9522.981445,NaN
364,9081.761719,NaN
365,9182.577148,NaN
366,9180.045898,NaN


## Section 5: Feature Data (X)
- Drops the Prediction column (the target) — leaving only features (Price, possibly others).

- Converts DataFrame to a NumPy array (x) — required by scikit-learn.

- Removes the last 30 rows (since those have NaN Prediction values).

In [6]:
x = np.array(df.drop(['Prediction'],axis=1))
x = x[:len(df)-predictionDays]
print(x)

[[ 7881.84668 ]
 [ 7987.371582]
 [ 8052.543945]
 [ 8673.21582 ]
 [ 8805.77832 ]
 [ 8719.961914]
 [ 8659.487305]
 [ 8319.472656]
 [ 8574.501953]
 [ 8564.016602]
 [ 8742.958008]
 [ 8208.995117]
 [ 7707.770996]
 [ 7824.231445]
 [ 7822.023438]
 [ 8043.951172]
 [ 7954.12793 ]
 [ 7688.077148]
 [ 8000.32959 ]
 [ 7927.714355]
 [ 8145.857422]
 [ 8230.923828]
 [ 8693.833008]
 [ 8838.375   ]
 [ 8994.488281]
 [ 9320.352539]
 [ 9081.762695]
 [ 9273.521484]
 [ 9527.160156]
 [10144.55664 ]
 [10701.69141 ]
 [10855.37109 ]
 [11011.10254 ]
 [11790.91699 ]
 [13016.23145 ]
 [11182.80664 ]
 [12407.33203 ]
 [11959.37109 ]
 [10817.15527 ]
 [10583.13477 ]
 [10801.67773 ]
 [11961.26953 ]
 [11215.4375  ]
 [10978.45996 ]
 [11208.55078 ]
 [11450.84668 ]
 [12285.95801 ]
 [12573.8125  ]
 [12156.5127  ]
 [11358.66211 ]
 [11815.98633 ]
 [11392.37891 ]
 [10256.05859 ]
 [10895.08984 ]
 [ 9477.641602]
 [ 9693.802734]
 [10666.48242 ]
 [10530.73242 ]
 [10767.13965 ]
 [10599.10547 ]
 [10343.10645 ]
 [ 9900.767578]
 [ 9811.

## Section 6: Target Data (y)
- Takes the Prediction column (your target).
- Removes the last 30 rows to match the size of x.

In [7]:
y = np.array(df['Prediction'])
y = y[:-predictionDays]
print(y)

[10701.69141  10855.37109  11011.10254  11790.91699  13016.23145
 11182.80664  12407.33203  11959.37109  10817.15527  10583.13477
 10801.67773  11961.26953  11215.4375   10978.45996  11208.55078
 11450.84668  12285.95801  12573.8125   12156.5127   11358.66211
 11815.98633  11392.37891  10256.05859  10895.08984   9477.641602
  9693.802734 10666.48242  10530.73242  10767.13965  10599.10547
 10343.10645   9900.767578  9811.925781  9911.841797  9870.303711
  9477.677734  9552.860352  9519.145508  9607.423828 10085.62793
 10399.66895  10518.17481  10821.72656  10970.18457  11805.65332
 11478.16895  11941.96875  11966.40723  11862.93652  11354.02441
 11523.5791   11382.61621  10895.83008  10051.7041   10311.5459
 10374.33887  10231.74414  10345.81055  10916.05371  10763.23242
 10138.04981  10131.05566  10407.96484  10159.96094  10138.51758
 10370.82031  10185.5       9754.422852  9510.200195  9598.173828
  9630.664063  9757.970703 10346.76074  10623.54004  10594.49316
 10575.5332   10353.302

## Section 7: Train/Test Split + Future Data Prep
- train_test_split randomly splits data into:
- - xtrain, ytrain → for model training (80%)
- - xtest, ytest → for model evaluation (20%)
- predictionDays_array = last 30 days of data (without Prediction column)
- These will be used to forecast the next 30 days.

In [9]:
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(x,y, test_size = 0.2)
predictionDays_array = np.array(df.drop(['Prediction'],axis=1))[-predictionDays:]
print(predictionDays_array)

[[7550.900879]
 [7569.936035]
 [7679.867188]
 [7795.601074]
 [7807.058594]
 [8801.038086]
 [8658.553711]
 [8864.766602]
 [8988.59668 ]
 [8897.46875 ]
 [8912.654297]
 [9003.070313]
 [9268.761719]
 [9951.518555]
 [9842.666016]
 [9593.896484]
 [8756.430664]
 [8601.795898]
 [8804.477539]
 [9269.987305]
 [9733.72168 ]
 [9328.197266]
 [9377.013672]
 [9670.739258]
 [9726.575195]
 [9729.038086]
 [9522.981445]
 [9081.761719]
 [9182.577148]
 [9180.045898]]


## Section 8: Model Setup and Training
- SVR = Support Vector Regression, part of the SVM family but for continuous values.
- kernel='rbf' → radial basis function kernel — good for non-linear relationships.
- C=1e3 → regularization parameter (higher = fits training data more tightly).
- gamma=0.00001 → controls how far the influence of a single training example reaches.
- .fit() → trains the model on your training data.

In [10]:
from sklearn.svm import SVR
svr_rbf = SVR(kernel='rbf', C=1e3, gamma=0.00001)
svr_rbf.fit(xtrain, ytrain)

,kernel,'rbf'
,degree,3
,gamma,1e-05
,coef0,0.0
,tol,0.001
,C,1000.0
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


## Section 9: Model Accuracy
- .score() returns the R² coefficient of determination (how well predictions match actuals).
- Value closer to 1 = good fit, near 0 = poor, negative = worse than random.

In [11]:
svr_rbf_confidence = svr_rbf.score(xtest,ytest)
print('SVR_RBF accuracy :',svr_rbf_confidence)

SVR_RBF accuracy : 0.17390696896100555


## Section 10: Testing Predictions
- .predict(xtest) generates predictions for the unseen test data.
- Prints both predictions and actual values (ytest) for comparison.

In [12]:
svm_prediction = svr_rbf.predict(xtest)
print(svm_prediction)
print()
print(ytest)

[ 8606.7410069   9089.36632873  9003.14136096  9063.69624353
  8291.29395574 10320.597976    8671.56187517  7981.07289706
  9075.59558011  8118.2605075   9025.35181733  8110.86747255
 10335.74729509  8126.87748715  9099.07339448  8609.65776817
  7037.55361104  7959.38325992  9062.01224153  9353.66031174
  9310.31281877 10385.79654849  8033.0532043   7851.17038877
  8197.93517647  8138.55694868  8211.2912564   9004.05363441
  9120.06273735  8213.05845062  8382.35451648  8011.6621255
  8041.97489897  7824.58814332  8767.5191011   8135.46584665
  8904.0643455   8125.86562782  7406.75239135  9337.15681863
 10348.00776827  8912.16737994  8019.87801465 10416.94087364
 10407.08378563  8434.92495402  9752.76565825  8683.17609055
  8922.11917151  9006.66122934  9261.74319452  9116.26228735
  8330.90431718  8910.60369333  8165.80493688  8556.2990294
  8706.87711881  8665.05182061  8345.86220973  8569.99208695
  9344.20037393  8104.95948452  8444.79986787 10123.58369545
  8810.78465385  9329.5944

## Section 11: Predicting the Future (Next 30 Days)
- Uses your last 30 days of data (predictionDays_array) to predict the next 30 days of prices.
- Prints both predicted future values and the last known data for context.

In [13]:
svm_prediction = svr_rbf.predict(predictionDays_array)
print(svm_prediction)
print()
print(df.tail(predictionDays))

[7903.72361246 7905.99787273 8198.32822607 8786.41888828 8842.3551543
 9107.05847177 9062.84334455 8983.12873477 8416.25115756 8870.99557989
 8808.216475   8332.65142062 7658.07578485 8169.14292602 8011.50368287
 8375.87963321 9127.05960721 9018.497373   9103.43789934 7661.35846247
 8054.05522149 7865.36132822 8072.73155947 8192.01510418 8066.55796485
 8062.13302527 8444.4266422  7896.81708306 7580.47619888 7583.21929152]

           Price  Prediction
337  7550.900879         NaN
338  7569.936035         NaN
339  7679.867188         NaN
340  7795.601074         NaN
341  7807.058594         NaN
342  8801.038086         NaN
343  8658.553711         NaN
344  8864.766602         NaN
345  8988.596680         NaN
346  8897.468750         NaN
347  8912.654297         NaN
348  9003.070313         NaN
349  9268.761719         NaN
350  9951.518555         NaN
351  9842.666016         NaN
352  9593.896484         NaN
353  8756.430664         NaN
354  8601.795898         NaN
355  8804.477539      